# 13 - Target Engineering

**Project:** AI Quantitative Trading & Strategy Optimisation System  
**Phase:** 6 — Feature Engineering and Market Regime Detection  
**Status:** Executed against the recorded local acquisitions on 14 November 2026; owner review is pending.

## 1. Objective and boundary

Define candidate future-return, direction and realised-volatility targets at the approved 5-, 10- and 20-session horizons. Each target is indexed by its forecast-origin close *t* and uses only prices after *t* to construct the outcome. Targets are stored separately from all explanatory features.

This notebook defines labels only. It does not train or evaluate forecasting models, select an optimal target, develop trading strategies, or simulate execution. A feature available at close *t* cannot be assumed tradable at that same close; execution must be no earlier than the next eligible session.

## 2. Target definitions

Let *Pₜ* be the configured `Close` at forecast origin and *Pₜ₊ₕ* the close after *h* subsequent observations.

- **Forward simple return:** *Rₜ,ₕ = Pₜ₊ₕ / Pₜ − 1*.
- **Directional label:** `+1` if *Rₜ,ₕ > δ*; `−1` if *Rₜ,ₕ < −δ*; otherwise `0` (neutral). The baseline δ = 0 makes exactly zero returns neutral. Candidate δ values 0.5% and 1.0% are sensitivity scenarios only; no neutral threshold is approved as optimal.
- **Forward realised volatility:** square root of the sum of squared one-observation log returns from *t+1* through *t+h*. This is cumulative realised variation over the horizon, not an annualised estimate.

The calculations use `Close`, consistently with the prior Phase 6 notebooks. Returns are price returns and exclude dividends. No GBP conversion is applied; the price ratio is scale invariant, but corporate-action and source-revision limitations remain.

## 3. Completeness, horizons and alignment

The implementation uses row shifts within each asset’s acquired daily observations. An output is missing if any price in the origin-to-endpoint path is missing; future volatility is missing if any of its *h* one-observation returns is missing. The final *h* origins in each series have no complete target and remain missing. No values are imputed.

The approved horizons are expressed as trading sessions, while this implementation counts acquired daily rows. The historical files have not been reindexed against a fresh exchange calendar in this notebook, so row counts are the operational proxy and calendar completeness remains a limitation. Targets at 5, 10 and 20 observations overlap: adjacent origins share future returns, up to *h−1* observations. Any later chronological evaluation must purge at least the maximum relevant horizon of training origins before a validation boundary; any additional embargo must follow the chosen validation design. This notebook does not define a split scheme.

## 4. Leakage controls

Negative shifts appear only inside the target builder and intentionally access future observations. The returned frame contains targets only and must never be concatenated into explanatory features. The origin index remains at *t*; the target end is *t+h*. No future target statistic is used to transform an input feature. Any normalisation, feature selection or threshold selection must later be fitted inside chronological training windows only.

## 5. Missingness, classes and unresolved choices

Missing targets are retained as missing rather than converted to neutral labels. Class balance is measured separately for each horizon and each neutral-threshold scenario; changing δ changes class prevalence and the meaning of a neutral label. The 0.5% and 1.0% thresholds are illustrative candidates, not cost-adjusted or economically approved values. Transaction costs, corporate actions, total-return targets and execution prices are outside this target-definition step.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "configs" / "assets").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.targets import DEFAULT_HORIZONS, build_forward_targets

ASSET_CONFIG_PATHS = sorted((PROJECT_ROOT / "configs" / "assets").glob("*.yaml"))
HORIZONS = DEFAULT_HORIZONS
NEUTRAL_THRESHOLDS = (0.0, 0.005, 0.01)
assert ASSET_CONFIG_PATHS, "No configured market assets were found."
print(f"Forecast horizons: {HORIZONS}")
print(f"Neutral-threshold scenarios: {NEUTRAL_THRESHOLDS}")

Forecast horizons: (5, 10, 20)
Neutral-threshold scenarios: (0.0, 0.005, 0.01)


## 6. Load configured closing prices

Each configured asset must resolve to exactly one acquired Parquet file. The notebook reads `Close` without modifying Raw files and keeps each asset’s native date index.

In [2]:
market_data = {}
source_inventory = []
for config_path in ASSET_CONFIG_PATHS:
    with config_path.open(encoding="utf-8") as config_file:
        config = yaml.safe_load(config_file)
    symbol = config["provider"]["symbol"]
    candidates = sorted(
        (PROJECT_ROOT / "data" / "raw" / "market" / "yahoo_finance" / symbol)
        .glob(f"acquisition_*/{symbol}.parquet")
    )
    if len(candidates) != 1:
        raise RuntimeError(f"Expected exactly one acquired Parquet file for {symbol}; found {len(candidates)}.")
    frame = pd.read_parquet(candidates[0])
    if "Date" in frame.columns:
        frame = frame.set_index("Date")
    frame.index = pd.to_datetime(frame.index)
    frame.index.name = "forecast_origin_observation_date"
    if not frame.index.is_monotonic_increasing or frame.index.has_duplicates:
        raise ValueError(f"{symbol} dates must be unique and chronological.")
    if "Close" not in frame.columns:
        raise ValueError(f"No Close field is available for {symbol}.")

    market_data[symbol] = frame[["Close"]]
    source_inventory.append({
        "symbol": symbol,
        "price_unit": config["asset"]["price_unit"],
        "price_unit_validation_status": config["asset"]["price_unit_validation_status"],
        "source_file": candidates[0].relative_to(PROJECT_ROOT).as_posix(),
        "observations": len(frame),
        "first_observation": frame.index.min(),
        "last_observation": frame.index.max(),
    })

source_inventory = pd.DataFrame(source_inventory).set_index("symbol")
source_inventory

,price_unit,price_unit_validation_status,source_file,observations,first_observation,last_observation
symbol,,,,,,
^FTSE,index_points,not_applicable,data/raw/market/yahoo_finance/^FTSE/acquisitio...,11004,1984-01-03,2026-11-02
NG.L,GBp,pending,data/raw/market/yahoo_finance/NG.L/acquisition...,7890,1995-12-11,2026-11-02
SSE.L,GBp,pending,data/raw/market/yahoo_finance/SSE.L/acquisitio...,9060,1991-06-17,2026-11-02
SVT.L,GBp,pending,data/raw/market/yahoo_finance/SVT.L/acquisitio...,9455,1989-12-11,2026-11-02


## 7. Construct separate target tables

Each returned frame contains targets only. Its index is the origin observation, not the future target end date. The target columns must remain separate from the feature frames created in Notebooks 10–12.

In [3]:
targets_by_asset = {
    symbol: build_forward_targets(
        frame,
        price_column="Close",
        horizons=HORIZONS,
        neutral_thresholds=NEUTRAL_THRESHOLDS,
    )
    for symbol, frame in market_data.items()
}

assert all(
    not any(column.startswith(("simple_return", "momentum", "sma_", "trend_")) for column in targets.columns)
    for targets in targets_by_asset.values()
)
print(f"Target tables created: {len(targets_by_asset)}; target columns per asset: {len(next(iter(targets_by_asset.values())).columns)}")

Target tables created: 4; target columns per asset: 15


## 8. Missingness and direction class-balance review

Class proportions exclude incomplete targets. They are descriptive diagnostics, not a basis for selecting a threshold or training a classifier.

In [4]:
class_balance_rows = []
missingness_rows = []
for symbol, targets in targets_by_asset.items():
    for column in targets.columns:
        missingness_rows.append({
            "symbol": symbol,
            "target": column,
            "missing_count": int(targets[column].isna().sum()),
            "valid_count": int(targets[column].notna().sum()),
        })
        if column.startswith("direction_"):
            counts = targets[column].value_counts(dropna=True)
            denominator = int(counts.sum())
            for label, name in [(-1, "negative"), (0, "neutral"), (1, "positive")]:
                class_balance_rows.append({
                    "symbol": symbol,
                    "target": column,
                    "class": name,
                    "count": int(counts.get(label, 0)),
                    "proportion": float(counts.get(label, 0) / denominator) if denominator else float("nan"),
                })

missingness_summary = pd.DataFrame(missingness_rows).set_index(["symbol", "target"])
class_balance = pd.DataFrame(class_balance_rows).set_index(["symbol", "target", "class"])
missingness_summary, class_balance

(                                       missing_count  valid_count
 symbol target                                                    
 ^FTSE  forward_return_5                          819        10185
        direction_5_0bp                           819        10185
        direction_5_50bp                          819        10185
        direction_5_100bp                         819        10185
        forward_realised_volatility_5             819        10185
        forward_return_10                        1385         9619
        direction_10_0bp                         1385         9619
        direction_10_50bp                        1385         9619
        direction_10_100bp                       1385         9619
        forward_realised_volatility_10           1385         9619
        forward_return_20                        2357         8647
        direction_20_0bp                         2357         8647
        direction_20_50bp                        2357         

In [5]:
# Inspect origin-aligned labels and their future end dates for one configured asset.
example_symbol = next(iter(targets_by_asset))
example_targets = targets_by_asset[example_symbol].copy()
example_targets["origin_date"] = example_targets.index
example_targets["20_observation_target_end"] = pd.Series(market_data[example_symbol].index, index=market_data[example_symbol].index).shift(-20)
example_targets.tail(25)

,forward_return_5,direction_5_0bp,direction_5_50bp,direction_5_100bp,forward_realised_volatility_5,forward_return_10,direction_10_0bp,direction_10_50bp,direction_10_100bp,forward_realised_volatility_10,forward_return_20,direction_20_0bp,direction_20_50bp,direction_20_100bp,forward_realised_volatility_20,origin_date,20_observation_target_end
forecast_origin_observation_date,,,,,,,,,,,,,,,,,
2026-09-29,-0.008931,-1,-1,0,0.018257,0.001279,1,0,0,0.024626,0.004682,1,0,0,0.032236,2026-09-29,2026-10-27
2026-09-30,-0.013907,-1,-1,-1,0.019692,0.011211,1,1,1,0.025432,0.008344,1,1,0,0.032115,2026-09-30,2026-10-28
2026-10-01,0.001275,1,0,0,0.010241,0.039518,1,1,1,0.021818,0.028173,1,1,1,0.027432,2026-10-01,2026-10-29
2026-10-02,0.008612,1,1,0,0.014328,0.040136,1,1,1,0.021914,0.025597,1,1,1,0.027251,2026-10-02,2026-10-30
2026-10-05,0.015089,1,1,1,0.017033,0.025415,1,1,1,0.024202,NaN,<NA>,<NA>,<NA>,NaN,2026-10-05,2026-11-02
2026-10-06,0.010302,1,1,1,0.016526,0.018147,1,1,1,0.024023,NaN,<NA>,<NA>,<NA>,NaN,2026-10-06,NaT
2026-10-07,0.025472,1,1,1,0.016094,0.028149,1,1,1,0.022754,NaN,<NA>,<NA>,<NA>,NaN,2026-10-07,NaT
2026-10-08,0.038193,1,1,1,0.019265,0.027141,1,1,1,0.022845,NaN,<NA>,<NA>,<NA>,NaN,2026-10-08,NaT
2026-10-09,0.031255,1,1,1,0.016581,0.013106,1,1,1,0.020531,NaN,<NA>,<NA>,<NA>,NaN,2026-10-09,NaT


## 9. Interpretation, limitations and later validation requirements

- Forward return and volatility columns deliberately contain future information. They are outcomes only and are prohibited from explanatory feature inputs.
- The targets use price returns, not dividend-inclusive total returns. Yahoo Finance adjustment and corporate-action limitations remain unresolved.
- The 0.5% and 1.0% neutral thresholds are exploratory candidates. Their class balance varies by asset and horizon; no threshold is approved.
- Overlapping labels induce dependence. Later validation must use chronological splits and purge overlapping target horizons; embargo length depends on the validation design.
- Horizons count acquired rows, which are used as a trading-session proxy. Calendar completeness and missing-source observations remain limitations.
- No forecast model, strategy, backtest, same-bar execution, or real-money workflow is implemented.

### Evidence status

- **Implemented in this change:** forward return, direction and future realised-volatility formulas and unit/edge tests.
- **Executed on 14 November 2026:** data loading, forward target construction, missingness summaries and directional class-balance scenarios; outputs are stored here. Target tests passed.
- **Not validated:** optimal target, neutral threshold, class balance on current data, dividend-inclusive target, or future predictive usefulness.
